# free-exercise-db Pipeline

Interactive runner for the enrichment pipeline. Run cells top-to-bottom for a full pipeline pass.
Individual steps can be re-run independently once prior steps have completed.

**Steps:**
1. Setup
2. Fetch — download upstream source
3. Preprocess — normalise muscle/equipment strings
4. Enrich — LLM classification via Claude
5. Check stale — vocabulary drift detection
6. Ingest — morph-KGC + enrichment merge → ingested.ttl
7. Validate — SHACL validation
8. Explore — query the ingested graph

## 1. Setup

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

# Resolve paths relative to this notebook
HERE = Path().resolve()  # sources/free-exercise-db/
ROOT = HERE.parent.parent  # project root

def run(cmd: list[str], **kwargs) -> subprocess.CompletedProcess:
    """Run a command from the project root, streaming output."""
    result = subprocess.run(
        [sys.executable] + cmd,
        cwd=ROOT,
        capture_output=False,
        **kwargs,
    )
    return result

print(f"Project root: {ROOT}")
print(f"Python: {sys.executable}")

## 2. Fetch

Downloads `exercises.json` from upstream GitHub. Skip if already present.

In [ ]:
raw_exercises = HERE / "raw" / "exercises.json"

if raw_exercises.exists():
    data = json.loads(raw_exercises.read_text())
    print(f"exercises.json already present — {len(data)} exercises. Skipping fetch.")
    print("To re-fetch, delete raw/exercises.json and re-run this cell.")
else:
    run(["sources/free-exercise-db/fetch.py"])

## 3. Preprocess

Applies crosswalk CSVs to normalise source muscle/equipment strings. Writes `raw/exercises_normalized.json`.

In [ ]:
run(["sources/free-exercise-db/preprocess.py"])

## 4. Enrich

LLM enrichment via Claude. Writes one JSON file per exercise to `enriched/`.
Resumes automatically — already-enriched exercises are skipped.

Adjust `LIMIT` and `CONCURRENCY` as needed. Set `LIMIT = None` to enrich all exercises.

**Rate limit guidance:** At `--concurrency 1` you stay well within the 30k token/min limit.
At `--concurrency 4` you may hit it periodically — tenacity handles retries automatically.

In [ ]:
LIMIT = 10          # number of exercises to enrich (None = all)
CONCURRENCY = 1     # parallel requests
RANDOM = True       # pick exercises at random

cmd = ["sources/free-exercise-db/enrich.py", f"--concurrency={CONCURRENCY}"]
if LIMIT:
    cmd += [f"--limit={LIMIT}"]
if RANDOM:
    cmd += ["--random"]

run(cmd)

In [ ]:
# Force re-enrich specific exercises by ID
FORCE_IDS = ["Barbell_Deadlift", "Plank"]  # edit as needed

run(["sources/free-exercise-db/enrich.py", "--force"] + FORCE_IDS)

In [ ]:
# Inspect enrichment coverage
enriched_dir = HERE / "enriched"
quarantine_dir = HERE / "quarantine"
total = len(json.loads((HERE / "raw" / "exercises.json").read_text()))
n_enriched = len(list(enriched_dir.glob("*.json"))) if enriched_dir.exists() else 0
n_quarantine = len(list(quarantine_dir.glob("*.json"))) if quarantine_dir.exists() else 0

print(f"Total exercises:  {total}")
print(f"Enriched:         {n_enriched}  ({n_enriched/total*100:.1f}%)")
print(f"Quarantined:      {n_quarantine}")
print(f"Pending:          {total - n_enriched - n_quarantine}")

## 5. Check Stale

Detects enriched exercises whose vocabulary version stamps are behind the current ontology.
Run after any vocabulary version bump.

In [ ]:
run(["sources/free-exercise-db/check_stale.py", "--verbose"])

## 6. Ingest

Runs morph-KGC, merges enrichment data, applies repair queries, writes `ingested.ttl`.

In [ ]:
run(["sources/free-exercise-db/ingest.py"])

## 7. Validate

SHACL validation of enriched exercises. Exit 0 = conforms.

In [ ]:
result = run(["sources/free-exercise-db/validate.py"])
print("\nExit code:", result.returncode)

In [ ]:
# Run the SHACL unit test suite
run(["test_shacl.py"])

## 8. Explore

Query the ingested graph with SPARQL.

In [ ]:
from rdflib import Graph

ingested_ttl = HERE / "ingested.ttl"
if not ingested_ttl.exists():
    print("ingested.ttl not found — run Ingest first.")
else:
    g = Graph()
    g.parse(ingested_ttl, format="turtle")
    print(f"Loaded {len(g)} triples from ingested.ttl")

In [ ]:
# Exercises by movement pattern
PATTERN = "HipHinge"  # edit as needed

q = f"""
PREFIX feg: <https://placeholder.url#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?name WHERE {{
    ?ex a feg:Exercise ;
        rdfs:label ?name ;
        feg:movementPattern feg:{PATTERN} .
}} ORDER BY ?name
"""
results = list(g.query(q))
print(f"{len(results)} exercises with pattern {PATTERN}:")
for row in results:
    print(" ", row.name)

In [ ]:
# Exercises targeting a specific muscle as PrimeMover
MUSCLE = "GluteusMaximus"  # edit as needed

q = f"""
PREFIX feg: <https://placeholder.url#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?name WHERE {{
    ?ex a feg:Exercise ;
        rdfs:label ?name ;
        feg:hasInvolvement ?inv .
    ?inv feg:muscle feg:{MUSCLE} ;
         feg:degree feg:PrimeMover .
}} ORDER BY ?name
"""
results = list(g.query(q))
print(f"{len(results)} exercises with {MUSCLE} as PrimeMover:")
for row in results:
    print(" ", row.name)

In [ ]:
# Enrichment coverage summary
q = """
PREFIX feg: <https://placeholder.url#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?pattern (COUNT(?ex) AS ?count) WHERE {
    ?ex a feg:Exercise ;
        feg:movementPattern ?pattern .
} GROUP BY ?pattern ORDER BY DESC(?count)
"""
results = list(g.query(q))
print("Movement pattern distribution:")
for row in results:
    label = str(row.pattern).split("#")[-1]
    print(f"  {label:30s} {row['count']}")

In [ ]:
# Quarantine review — exercises that failed enrichment
quarantine_dir = HERE / "quarantine"
quarantine_files = sorted(quarantine_dir.glob("*.json")) if quarantine_dir.exists() else []

if not quarantine_files:
    print("No quarantined exercises.")
else:
    print(f"{len(quarantine_files)} quarantined exercises:")
    for f in quarantine_files:
        entry = json.loads(f.read_text())
        print(f"  {entry['exercise']['name']}: {entry['error'][:80]}")